In [2]:
# Create customer master data and order history for lifetime summaries
import pandas as pd
customers = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C04"],
    "signup_date": ["2025-11-01", "2026-01-15", "2026-02-10", "2026-02-20"],
    "segment": ["new", "returning", "new", "business"],
})

orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6],
    "customer_id": ["C01", "C01", "C02", "C03", "C03", "C05"],
    "order_date": [
        "2025-12-01", "2026-01-10", "2026-02-01",
        "2026-02-15", "2026-03-01", "2026-02-20",
    ],
    "amount": [100, 150, 200, 50, 75, 500],
    "status": ["complete", "complete", "refunded", "complete", "complete", "complete"],
})

In [ ]:
# Copy inputs so the pipeline does not mutate the caller
def customer_lifetime_pipeline(customers,orders):
    # Work on copies so the original inputs remain unchanged.
    customers=customers.copy()
    orders=orders.copy()

    # Parse date columns before calculating intervals.
    customers["signup_date"]=pd.to_datetime(customers["signup_date"],errors="coerce")
    orders["order_date"]=pd.to_datetime(orders["order_date"],errors="coerce")

    # Exclude refunded orders from lifetime revenue metrics.
    valid_orders=orders[orders["status"].str.lower().fillna("")!="refunded"].copy()

    valid_customers=customers.dropna(subset=['customer_id','signup_date'])
    valid_orders=valid_orders.dropna(subset=['order_id','customer_id','order_date','amount'])
    valid_orders=valid_orders[valid_orders["amount"]>0]

    # Keep unmatched orders for data-quality review instead of silently dropping them.
    unmatched_orders=valid_orders[
        ~valid_orders["customer_id"].isin(valid_customers["customer_id"])
    ]
    matched_orders=valid_orders[
        valid_orders["customer_id"].isin(valid_customers["customer_id"])
    ]

    # Aggregate completed, matched orders at customer level.
    customers_stats=matched_orders.groupby('customer_id').agg(
        total_orders=('order_id','nunique'),
        total_revenue=('amount','sum'),
        average_order_value=('amount','mean'),
        first_order_date=('order_date','min'),
    ).reset_index()

    # Left join keeps customers with no qualifying orders.
    customer_summary=valid_customers.merge(customers_stats,on='customer_id',how='left').fillna({
        "total_orders":0,
        "total_revenue":0,
        "average_order_value":0
    })

    customer_summary["days_from_signup_to_first_order"]=(
        customer_summary["first_order_date"]-customer_summary["signup_date"]
    ).dt.days
    return customer_summary,unmatched_orders


In [8]:
# Run the pipeline and inspect matched customers plus unmatched orders
customer_summary,unmatched_orders=customer_lifetime_pipeline(customers,orders)
print(f"Customer's summary\n{'-'*80}\n",customer_summary)
print(f"Unmatched orders\n{'-'*80}\n",unmatched_orders)


Customer's summary
--------------------------------------------------------------------------------
   customer_id signup_date    segment  total_orders  total_revenue  \
0         C01  2025-11-01        new           2.0          250.0   
1         C02  2026-01-15  returning           0.0            0.0   
2         C03  2026-02-10        new           2.0          125.0   
3         C04  2026-02-20   business           0.0            0.0   

   average_order_value first_order_date  days_from_signup_to_first_order  
0                125.0       2025-12-01                             30.0  
1                  0.0              NaT                              NaN  
2                 62.5       2026-02-15                              5.0  
3                  0.0              NaT                              NaN  
Unmatched orders
--------------------------------------------------------------------------------
    order_id customer_id order_date  amount    status
5         6         C05 20

### Important notes

- The function copies its inputs, making the pipeline safer to reuse without changing the original DataFrames.
- Refunded, invalid, non-positive, and unmatched orders are excluded from customer metrics but unmatched orders are returned for review.
- A left join preserves customers with no qualifying orders; their aggregate metrics are filled with zero while their first-order date remains missing.
- `nunique()` prevents duplicate order rows from inflating the order count, while revenue still sums each retained row.
- Validate that order dates are not earlier than signup dates and define how duplicate IDs, cancellations, and partial refunds should be handled.
- Lifetime summaries depend on the observation window; recent customers may appear to have lower totals simply because they have had less time to order.
